# Download of the BnF text data using their internal API

There are around 120 newspaper titles which we wish to download from the BnF (Bibliothèque nationale de France) using their internal API. 

This notebook is a debug approach to understand how this download can be scripted and automated, to be applied to any subset of the new titles to download.

## Imports and setup

In [4]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd
import json
import numpy as np

from impresso_essentials.utils import ALL_MEDIA

load_dotenv()

False

In [ ]:
# ensure the API credentials are in the environment
secret = os.environ["BNF_API_SECRET"]

### Add the main endpoints provided by the BnF

Cela permet de récupérer une réponse JSON de ce type :

{
"access_token": "TOKEN",
"scope": "default",
"token_type": "Bearer",
"expires_in": 3600
}

Il faudra ensuite récupérer la valeur de "access_token" pour l’injecter dans chaque appel d’API, dans un header "Authorization" dont la valeur est préfixée par "token_type", puis suivie de la valeur du token.

Dans "expires_in", on a la durée de validité du token en secondes.

Si j’appelle une URL IIIF, par exemple :

curl -X 'GET' \
'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4600049g/manifest.json'\
-H 'Authorization: Bearer TOKEN'


À noter : pour signaler les dépassements de quota, l’erreur HTTP retournée est 429. Dans ce cas, nous renvoyons une réponse avec le JSON suivant :

{"code":"900804","message":"Message throttled out","description":"You have exceeded your quota. You can access API after 2026-avr.-22 15:31:00+0000 UTC","nextAccessTime":"2026-avr.-22 15:31:00+0000 UTC"}

et notamment la propriété "nextAccessTime", qui nous indique quand nous pourrons refaire des requêtes.

Nous renvoyons également un header "retry-after" dans la réponse http, exemple :
retry-after: mer., 26 avr. 2026 15:31:00 GMT

qui donne cette information de reprise possible. Attention, l’heure est exprimée en GMT.

L'idée serait d'appeler le manifest json par l'API IIIF pour chaque document de la liste d'ark fournie par Antoine, ou récupérer par l'API date_periodique_gallica ("/Issues" )


https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4600049g/manifest.json'

exemple : avec le manifest ci-dessus.
        L'objet metadata contient les métadonnées biblio (équivalent à OAIRecord)
        L'objet items, est un tableau dont chaque objet représente une page et ses informations, (on a donc la structure du document)
                vous y verrez la taille master (propriété height et width),
                l'objet seeAlso, qui si il y a un ocr contient un objet dont le label est "ALTOXML" et l'id contient une url finissant par alto.xml, qui est un lien de récupération                         directe de l'OCR au format ALTO

Les routes dans ce manifest sont préfixé avec openapi.bnf.fr, pour suivre les liens dans ce manifest, si vous ne créér pas vous même les appels, il faudra remplacer openapi.bnf.fr par openapiproext.bnf.fr et signer avec le token chaque appel.


Les APIs, accessibles pour le moment sont les suivantes :


l'équivalent de /services/Issues documenté sur api.bnf.fr

l'api IIIF v3  documentée dans son implémentation ici pour l'api image https://iiif.io/api/image/3.0/et ici pour la présentation https://iiif.io/api/presentation/3.0/


#### Fetch the access token

In [5]:
# url to get the token
TOKEN_URL = "https://apimauthproext.bnf.fr/oauth2/token"

# to be used as follows: 
# curl -X POST "https://apimauthproext.bnf.fr/oauth2/token" -u "KEY:SECRET" -d "grant_type=client_credentials"

def get_access_token(token_url=TOKEN_URL):
    key = os.environ["BNF_API_KEY"]
    secret = os.environ["BNF_API_SECRET"]
    response = requests.post(
        token_url,
        auth=(key, secret),
        data={"grant_type": "client_credentials"},
    )
    response.raise_for_status()
    return response.json()


In [ ]:
response = get_access_token()
response

In [7]:
token = response["access_token"]

In [38]:
IIIF_PRES_URL = "https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/{ark_id}/manifest.json"

IIIF_PRES_URL.format(ark_id='bpt6k4600049g')

'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4600049g/manifest.json'

In [8]:
IIIF_PRES_URL = "https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/{ark_id}/manifest.json"

def get_iiif_presentation_for_ark(ark_id, token=token, iiif_pres_url=IIIF_PRES_URL):
    url = iiif_pres_url.format(ark_id=ark_id)
    response = requests.get(
        url,
        headers={"Authorization": f"Bearer {token}"},
    )
    response.raise_for_status()
    return response.json()


In [9]:
response = get_iiif_presentation_for_ark('bpt6k4690357j')
response

{'@context': 'http://iiif.io/api/presentation/3/context.json',
 'id': 'https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/manifest.json',
 'type': 'Manifest',
 'label': {'fr': ['BnF, département Droit, économie, politique, JO-96197']},
 'provider': [{'id': 'https://gallica.bnf.fr/',
   'type': 'Agent',
   'label': {'fr': ['BNF Gallica']},
   'homepage': [{'id': 'https://gallica.bnf.fr',
     'type': 'Text',
     'label': {'fr': ["Page d'accueil Gallica"],
      'en': ['Homepage for BNF Gallica Project']}}],
   'logo': [{'id': 'https://gallica.bnf.fr/mbImage/logos/logo-bnf.png',
     'type': 'Image',
     'format': 'image/png'}],
   'seeAlso': [{'id': 'https://gallica.bnf.fr/edit/und/conditions-dutilisation-des-contenus-de-gallica',
     'type': 'Text',
     'label': {'fr': ["Conditions d'utilisation des contenus de Gallica"],
      'en': ["Gallica's contents terms and conditions of use"]}}]}],
 'behavior': ['paged'],
 'homepage': [{'id': 'https://gallica.bnf.fr/ark:/1

In [11]:
page_altos_xmls = []
for entry_id, m_entry in enumerate(response['metadata']):
    if m_entry['label']['fr'][0] == 'Date':
        print(f"Date: {m_entry['value']['fr'][0]}")
    if m_entry['label']['fr'][0] == 'Langue':
        print(f"Language: {m_entry['value']['fr'][0]}")
    if m_entry['label']['fr'][0] == 'Titre':
        print(f"Title: {m_entry['value']['fr'][0]}")
for entry_id, i_entry in enumerate(response['items']):
    print(f"page {entry_id+1}:")
    for elem in i_entry['seeAlso']:
        print(f"id: {elem['id']}")
        page_altos_xmls.append(elem['id'].replace("openapi.bnf.fr", "openapiproext.bnf.fr"))

Language: Français
Title: Le Petit Marocain
Date: 1934-01-09
page 1:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f1/alto.xml
page 2:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f2/alto.xml
page 3:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f3/alto.xml
page 4:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f4/alto.xml
page 5:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f5/alto.xml
page 6:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f6/alto.xml
page 7:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f7/alto.xml
page 8:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f8/alto.xml
page 9:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f9/alto.xml
page 10:
id: https://openapi.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f10/alto.xml


In [12]:
page_altos_xmls

['https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f1/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f2/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f3/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f4/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f5/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f6/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f7/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f8/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f9/alto.xml',
 'https://openapiproext.bnf.fr/iiif/presentation/v3/ark:/12148/bpt6k4690357j/f10/alto.xml']

In [ ]:

response = requests.get(
        url,
        headers={"Authorization": f"Bearer {token}"},
    )

## Load the lists of ark identifiers to download

We have multiple lists of ark identifiers:
- One with the ark identifier for each newspaper title we want to download, `docs_per_ark_bib.csv`, which needs to be mapped to its title using the `03_BNF-Media-List.csv` file which we provided them in the first place.
- One per title, with the ark identifier for each issue of the newspaper, which is named after the ark identifier of the title in question. All of these lists are in the `impresso-text-acquisition/text_preparation/data/sample_data/BNF_API/BnF_API_info/arks_num_per_ark_bib` folder.


The first step is thus to perform a light preprocessing:
- link the list of newspapers (with its arks) to the list of arks mapped to the number of issues, and exclude the titles for which we already have the data (the ones which have an alias).
- devise unique aliases for each title, to be used as the name of the folder where the data will be downloaded.
- map each title to its list of issues, either on the fly in the script or storing the mapping in a dictionary, to be used directly for the download of the data.

In [17]:
MEDIA_LIST_FILE = "../text_preparation/data/sample_data/BNF_API/BnF_API_info/03_BNF-Media-List.csv"
DOCS_PER_ARK_FILE = "../text_preparation/data/sample_data/BNF_API/BnF_API_info/docs_per_ark_bib.csv"

ISSUE_ARKS_PER_TITLE_DIR = "../text_preparation/data/sample_data/BNF_API/BnF_API_info/arks_num_per_ark_bib"

### 1. prep of the DF with the ark ids and number of documents

This will also ensure us to know exactly for which title we will download data, and the amount of issues to download for each title.

In [32]:
# First preparation for the media list file: only keep alias, ark_id, dates at first
media_list_df = pd.read_csv(MEDIA_LIST_FILE, header=1)

columns_renaming = {
    "Included Time Period START DATE (1st January) (to be filled only if it applies)": "start_year", 
    "Included Time Period END DATE (31st December) (to be filled only if it applies)": "end_year"
}
media_list_df = media_list_df.rename(columns=columns_renaming)

columns_to_keep = ['Alias', 'Title', 'ARK ID', 'start_year', 'end_year']
media_list_df = media_list_df[columns_to_keep]

media_list_df.start_year.astype(int)
media_list_df.end_year.astype(int)

media_list_df

,Alias,Title,ARK ID,start_year,end_year
0,excelsior,Excelsior,cb32771891w,1910,1920
1,lafronde,La Fronde,cb327788531,1897,1899
2,lafronde,La Fronde,cb327788531,1900,1929
3,marieclaire,Marie-Claire,cb343488519,1937,1944
4,oeuvre,Oeuvre,cb34429265b,1915,1944
...,...,...,...,...,...
173,NaN,Revue de l'histoire des colonies françaises,cb32857372f,1913,1931
174,NaN,Revue des questions coloniales et maritimes,cb32858753h,1914,1938
175,NaN,Les Tablettes coloniales,cb328754761,1888,1891
176,NaN,Togo Cameroun (Paris),cb34407680f,1929,1937


In [37]:
# now groupby Alias, Title and ArkID and to keep the first start year and last end year

bnf_titles_df = media_list_df.groupby(['Title', 'ARK ID']).agg({"Alias": set, "start_year": min, "end_year": max}).reset_index()
bnf_titles_df

,Title,ARK ID,Alias,start_year,end_year
0,Abendland,cb32680789v,{nan},1925,1930
1,Armée coloniale (L'),cb327021687,{nan},1891,1892
2,Bulletin colonial,cb327173415,{nan},1836,1849
3,Bulletin de l'Office colonial,cb34378143c,{nan},1908,1919
4,Bulletin de la Société de géographie et d'étud...,cb37131292d,{nan},1902,1937
...,...,...,...,...,...
134,"The Chicago tribune and the Daily news, New Yo...",cb327410645,{nan},1923,1934
135,The New York Herald,cb391150993,{nan},1887,1935
136,The Paris tribune,cb328330320,{nan},1924,1937
137,Togo Cameroun (Paris),cb34407680f,{nan},1929,1937


In [40]:
# Now load the other DF with the ARK IDs and counts, and merge it
arks_nums_df = pd.read_csv(DOCS_PER_ARK_FILE, sep=';', header=0)
arks_nums_df

,ARK BIB,DOCS
0,cb32777702m,101
1,cb32750077g,28
2,cb328066631,52917
3,cb32737062q,1037
4,cb32778606z,47
...,...,...
121,cb327071375,16415
122,cb32769854d,7
123,cb327596899,16867
124,cb32744363c,14


In [77]:
titles_to_dl_df = bnf_titles_df.merge(arks_nums_df, how='left', left_on="ARK ID", right_on="ARK BIB")
titles_to_dl_df

,Title,ARK ID,Alias,start_year,end_year,ARK BIB,DOCS
0,Abendland,cb32680789v,{nan},1925,1930,cb32680789v,19.0
1,Armée coloniale (L'),cb327021687,{nan},1891,1892,cb327021687,59.0
2,Bulletin colonial,cb327173415,{nan},1836,1849,cb327173415,99.0
3,Bulletin de l'Office colonial,cb34378143c,{nan},1908,1919,cb34378143c,15.0
4,Bulletin de la Société de géographie et d'étud...,cb37131292d,{nan},1902,1937,cb37131292d,27.0
...,...,...,...,...,...,...,...
134,"The Chicago tribune and the Daily news, New Yo...",cb327410645,{nan},1923,1934,cb327410645,4260.0
135,The New York Herald,cb391150993,{nan},1887,1935,cb391150993,17385.0
136,The Paris tribune,cb328330320,{nan},1924,1937,cb328330320,8.0
137,Togo Cameroun (Paris),cb34407680f,{nan},1929,1937,cb34407680f,31.0


From this df we can see that indeed the list of ARK identifiers is complete, with only the titles which we already have which are missing. 
One specificity to note is that the title "Oeuvre" is already present in our data (has an alias), but it is also present in the list of ark identifiers to download. This is because we only have the data starting in 1914 and that we wish to also collect it for the years 1904-1914.

We will see if we download the data for the full period or only for the missing years, but for now we will consider the full period.

The final step before saving this csv is: 
- removing all the rows which have an alias already in the "Alias" column, as we already have the data for those titles.
- generating a unique alias for each title, to be used as the name of the folder where the data will be downloaded, and verifying that it's not already used in our collection of aliases.

In [78]:
titles_to_dl_df = titles_to_dl_df[titles_to_dl_df['Alias'] == {np.nan}]
titles_to_dl_df.DOCS = titles_to_dl_df.DOCS.astype(int)
titles_to_dl_df.drop("ARK BIB", axis=1, inplace=True)
titles_to_dl_df

,Title,ARK ID,Alias,start_year,end_year,DOCS
0,Abendland,cb32680789v,{nan},1925,1930,19
1,Armée coloniale (L'),cb327021687,{nan},1891,1892,59
2,Bulletin colonial,cb327173415,{nan},1836,1849,99
3,Bulletin de l'Office colonial,cb34378143c,{nan},1908,1919,15
4,Bulletin de la Société de géographie et d'étud...,cb37131292d,{nan},1902,1937,27
...,...,...,...,...,...,...
134,"The Chicago tribune and the Daily news, New Yo...",cb327410645,{nan},1923,1934,4260
135,The New York Herald,cb391150993,{nan},1887,1935,17385
136,The Paris tribune,cb328330320,{nan},1924,1937,8
137,Togo Cameroun (Paris),cb34407680f,{nan},1929,1937,31


In [ ]:
ARK_ID_TO_ALIAS = {
    "cb32680789v": "abendland",
    "cb327021687": "armeecoloniale",
    "cb327173415": "bulletincol",
    "cb34378143c": "bulletinoffcol",
    "cb37131292d": "bsgecm",
    "cb32724358c": "bsecm",
    "cb327272204": "bcaf",
    "cb32732353c": "bocfo",
    "cb32737062q": "candide",
    "cb32738400h": "cesoir",
    "cb32742299m": "cinejournal",
    "cb34501455d": "combat",
    "cb32751516w": "courriermarna",
    "cb34477322w": "demokratischeztg",
    "cb32887413t": "vsve",
    "cb328306427": "elouma",
    "cb32771225f": "europecolonies",
    "cb44498563s": "francelibre",
    "cb32777450n": "franceboheme",
    "cb32777702m": "franceeuropeor",
    "cb327781475": "francerussie",
    "cb327782343": "franceyougoslavie",
    "cb32778160c": "francesoir",
    "cb32778606z": "freeeurope",
    "cb32784069f": "gringoire",
    "cb391150993": "herald",           # International Herald Tribune / The New York Herald (same ARK)
    "cb344295535": "actionfrancaise1899",
    "cb326819451": "actionfrancaise1908",
    "cb326834694": "lafrique1844",
    "cb34440169p": "alsacefrancaise",
    "cb34371852k": "aurore1943",
    "cb327071375": "lauto",
    "cb327596899": "echoalger",
    "cb32759772v": "echoran",
    "cb327595295": "echangouleme",
    "cb32771219h": "europeartistique",
    "cb327712243": "europecoloniale",
    "cb32771226s": "europedabord",
    "cb327712274": "europedanubienne",
    "cb32771243c": "europefinanciere1",
    "cb32771244q": "europefinanciere2",
    "cb327712510": "europefuture",
    "cb32771252b": "europeill",
    "cb32771253p": "europeillrev",
    "cb32771256q": "europeindcom",
    "cb32771272z": "europenouvelle",
    "cb32771280k": "europeor1919",
    "cb32771279c": "europeorroum",
    "cb327712003": "europejq",
    "cb32771204g": "europepscil",
    "cb327877302": "humanite",
    "cb32788323x": "ikdam",
    "cb32793876w": "intransigeant",
    "cb328840924": "unionfrancaise",
    "cb34520232c": "univers",
    "cb34429265b": "oeuvre",        # same ARK as existing alias, different time window
    "cb32740226x": "lacharente",
    "cb343631418": "lacroix",
    "cb45584487n": "defensenationalparis",
    "cb32755585p": "democratiepacifique",
    "cb327558876": "depechetoulouse",
    "cb327559237": "depechecolill",
    "cb327773077": "lafranceparis",
    "cb44403307p": "gazettecoloniale",
    "cb32784054d": "lagrimace",
    "cb34425747t": "jeuneeurope1930",
    "cb32795778s": "jeuneeuroperev",
    "cb32802914p": "lajustice",
    "cb328066631": "laliberte",
    "cb32806743p": "libertecol",
    "cb328261098": "nouvellefrancemars",
    "cb32837965d": "petitepresse",
    "cb34448033b": "lapresse",
    "cb328424860": "pressecolill",
    "cb34425265c": "quinzainecol",
    "cb328507767": "renaissancecol",
    "cb328569480": "revueorienth",
    "cb32889084q": "vielatine",
    "cb34431415m": "bonnetrouge",
    "cb327152851": "bouffon",
    "cb34348420x": "canardenchaine",
    "cb32738229p": "lecaucase",
    "cb32747578p": "leconstitutionnel",
    "cb34448436g": "lecorsaire",
    "cb32749956z": "lecourrier",
    "cb32750077g": "courriercolill",
    "cb32750643t": "courrierlondres",
    "cb327524145": "cridespeuples",
    "cb32752488q": "cripeuple1871",
    "cb344484501": "figaro1826",
    "cb344551004": "figaro1839",
    "cb34355551z": "figaro1854",
    "cb343599097": "figarosupl",
    "cb32777201w": "franctireur",
    "cb32783482h": "grandechonord",
    "cb34473289x": "lejournal",
    "cb34459430v": "mondecolill",
    "cb328343740": "lepays",
    "cb32895690j": "lepetitjournal",
    "cb344696449": "petitmarocain",
    "cb34348990g": "lepeuple",
    "cb32843836s": "progrescol",
    "cb34431794k": "letemps",
    "cb326934111": "annalescol",
    "cb32709664x": "lesbalkans",
    "cb32744363c": "lescolonies",
    "cb32769854d": "etatsuniseurope",
    "cb34348821c": "lettresfrancaises",
    "cb328754761": "tablettescol",
    "cb32806572q": "liberation",
    "cb32825412r": "notretemps",
    "cb32826916r": "nouvellestcheco",
    "cb41193663x": "oenantes",
    "cb328314062": "paixtravail",
    "cb32832208f": "parisbalkans",
    "cb32832453t": "pariseurope",
    "cb32832672n": "parismidi",
    "cb343985327": "revuecol",
    "cb328566125": "revuecolan",
    "cb32857372f": "rhcf",
    "cb32858753h": "rqcm",
    "cb32876855d": "terreeurope",
    "cb327410645": "chicagotribune",
    "cb328330320": "paristribune",
    "cb34407680f": "togocameroun",
    "cb32887275t": "vendredi",
}


In [ ]:
# checking none are already in the data

for alias in ARK_ID_TO_ALIAS.values():
    if alias in ALL_MEDIA:
        print(f"Alias {alias} is already in all media!!")

# there should only be oeuvre popping up

Alias oeuvre is already in all media!!


Now remove duplicated ark IDs, ad assign the new ones to the data and save

In [80]:
titles_to_dl_df[titles_to_dl_df['ARK ID'].duplicated()]

,Title,ARK ID,Alias,start_year,end_year,DOCS
60,La Croix (1880),cb343631418,{nan},1880,1953,29401
135,The New York Herald,cb391150993,{nan},1887,1935,17385


In [82]:
# two duplicated titles (arks) to be removed: 
#  - cb343631418 ("La Croix (1880)" and "La Croix") and 
#  - cb391150993 (International Herald Tribune / The New York Herald)

titles_to_dl_df.drop_duplicates(subset='ARK ID', inplace=True)
titles_to_dl_df['Alias'] = titles_to_dl_df['ARK ID'].apply(lambda x: ARK_ID_TO_ALIAS[x])

titles_to_dl_df

,Title,ARK ID,Alias,start_year,end_year,DOCS
0,Abendland,cb32680789v,abendland,1925,1930,19
1,Armée coloniale (L'),cb327021687,armeecoloniale,1891,1892,59
2,Bulletin colonial,cb327173415,bulletincol,1836,1849,99
3,Bulletin de l'Office colonial,cb34378143c,bulletinoffcol,1908,1919,15
4,Bulletin de la Société de géographie et d'étud...,cb37131292d,bsgecm,1902,1937,27
...,...,...,...,...,...,...
133,Terre d'Europe. Revue,cb32876855d,terreeurope,1933,1940,8
134,"The Chicago tribune and the Daily news, New Yo...",cb327410645,chicagotribune,1923,1934,4260
136,The Paris tribune,cb328330320,paristribune,1924,1937,8
137,Togo Cameroun (Paris),cb34407680f,togocameroun,1929,1937,31


In [83]:
# save the result
OUT_BNF_TITLES_FILE = "../text_preparation/data/sample_data/BNF_API/BnF_API_info/titles_to_download.csv"

titles_to_dl_df.to_csv(OUT_BNF_TITLES_FILE, index=True)